# Capstone A - Customer Support Routing with Zero Training

*Part A capstone - ML and NLP by Data Trainers LLC.*

This capstone is the payoff for Part A. In A1 you did NLP by hand with spaCy and gensim. In A2 you watched pretrained transformers solve task after task in three lines with `pipeline()`. Now you put it all together on a real, noisy, production-shaped problem and ship a working solution with **zero training and zero labelled data**.

## Learning objectives

By the end you will be able to:

1. Load and sanity-check a real customer-support corpus (TWCS).
2. Recover weak labels from implicit reply metadata (the labels are not a column in the file).
3. Define human-readable routing categories and classify tweets with `pipeline("zero-shot-classification")` - no training.
4. Extract product and account mentions with `pipeline("ner")` as routing metadata.
5. Evaluate routing quality qualitatively and against the weak labels.
6. Add a confidence threshold so low-confidence tweets fall back to a human.
7. Wrap the router in a tiny Gradio demo and reason about when zero-shot is not enough.

## Prerequisites

- A1 (spaCy NER, gensim topics) and A2 (the `pipeline()` tour). You should already have called `pipeline("zero-shot-classification")` and `pipeline("ner", aggregation_strategy="simple")`.

## Session format

Roughly 60-90 minutes: short theory, a runnable demo, then a hands-on lab for each section. Runs on Google Colab (GPU optional; everything here also works on CPU, just slower).

## Section 0: Environment Setup

This notebook uses **HuggingFace `transformers` pipelines** on top of PyTorch. We do not train anything; every model is pretrained and downloaded from the HuggingFace Hub. Run the next two cells once per Colab session.

In [ ]:
# Install required packages (run this first in Google Colab).
# numpy<2 is pinned because several pretrained-model wheels are not yet built against numpy 2.x.
!pip install -q "numpy<2" "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30" \
    pandas matplotlib seaborn

# spaCy is already used in A1; we only need transformers' own NER here, so spaCy is optional.
print("Install step done. If Colab asks you to restart the runtime, do it, then re-run from here.")

In [ ]:
# Core imports
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
from transformers import pipeline

# Reproducibility (sampling is the only randomness here; the models are deterministic at inference)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(42)

# Device: pipelines take an int device id. 0 = first GPU, -1 = CPU.
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"transformers will run on: {'GPU (cuda:0)' if DEVICE == 0 else 'CPU'}")
print(f"PyTorch version: {torch.__version__}")
print("Environment ready.")

## Section 1: Load the TWCS Dataset

The **Twitter Customer Support (TWCS)** corpus is millions of tweets between customers and company support accounts. We will use a subsample for speed; the methodology scales up unchanged.

Each row is a single tweet with these columns:

| Column | Meaning |
|--------|---------|
| `tweet_id` | Unique id for the tweet |
| `author_id` | Username (company, e.g. `AppleSupport`) or a numeric string (a customer) |
| `inbound` | `True` if the tweet is from a customer to a company, `False` if from the company |
| `created_at` | Timestamp |
| `text` | The tweet text |
| `response_tweet_id` | Comma-separated reply `tweet_id`s, or empty |
| `in_response_to_tweet_id` | If this tweet is a reply, the `tweet_id` it replies to |

Notice there is **no routing column**. We will recover weak labels from the reply structure in a moment. But first: the whole point of Part A is that we can route tweets even without those labels.

In [ ]:
# Course-hosted copy of TWCS (real company handles preserved). dl=1 streams the raw CSV.
TWCS_URL = "https://www.dropbox.com/scl/fi/6w2dchvsthwdol934nprt/twcs.csv?rlkey=u5205iehhrog88qt8iwm4udxs&dl=1"

# Read only the first chunk of rows: enough to see every major company, fast to download.
SUBSET_SIZE = 80_000
twcs = pd.read_csv(TWCS_URL, nrows=SUBSET_SIZE)

print(f"Loaded {len(twcs):,} tweets")
print(f"Columns: {list(twcs.columns)}")
display(twcs.head(3))

# Sanity check: how many tweets are customer-to-company (inbound) vs company replies (outbound)?
print("\ninbound value counts (True = customer tweet, the thing we will route):")
print(twcs['inbound'].value_counts())

print("\nTop 10 author_ids (company support accounts dominate the outbound side):")
print(twcs['author_id'].value_counts().head(10))

## Section 2: Recover Weak Labels From Reply Metadata

Here is the whole capstone in five moves, each a short lab:

1. **Recover weak labels** from reply metadata, so we have something to spot-check against later. These are *weak* labels (the company that happened to reply), not a routing ground truth.
2. **Define routing categories** as plain-English `candidate_labels` (billing, technical, etc.).
3. **Route every tweet** with `pipeline("zero-shot-classification")` - the no-training superpower.
4. **Extract metadata** (products, account mentions) with `pipeline("ner")`.
5. **Evaluate and add a safety net**: a confidence threshold that sends unsure tweets to a human.

Not one line of model training in the whole notebook. That is the Part A promise. Let us start with move 1.

The dataset has no routing column, but it does encode *which company handled each tweet*: the reply to a customer tweet was authored by that company's support account. We can walk the reply links backwards to recover the handling company:

```
Customer tweet (inbound=True)   <-- we want to route this
        ^
        | in_response_to_tweet_id
        |
Company reply (inbound=False)   <-- its author_id tells us who handled it
```

We will use this as a **weak label**: useful to spot-check our zero-shot router, but not a true routing label (the company that replied is not the same thing as the right queue). This is also a real, reusable skill: recovering labels from implicit structure when nobody handed you a clean `label` column.

**Figure: recovering a weak company label by walking the reply link backwards.**

```mermaid
graph TD
    A[Customer tweet inbound True] -->|tweet_id| B[Company reply inbound False]
    B -->|in_response_to_tweet_id points back| A
    B --> C[author_id of reply]
    C --> D[Build reply_map parent_id to company]
    D --> E[Map onto inbound tweets]
    E --> F[Weak company label for spot-check]
```


In [ ]:
# Demo: recover the handling company for each customer tweet, end to end.

# The companies we will spot-check on (the busiest support accounts in this subsample).
TOP_AUTHORS = ['AppleSupport', 'AmazonHelp', 'SpotifyCares', 'Uber_Support']

# 1. Split customer tweets (inbound) from company replies (outbound).
inbound = twcs[twcs['inbound'] == True].copy()
outbound = twcs[twcs['inbound'] == False].copy()

# 2. Keep only replies authored by one of our target companies, with a valid parent tweet id.
outbound_targeted = outbound[outbound['author_id'].isin(TOP_AUTHORS)].copy()
outbound_targeted = outbound_targeted.dropna(subset=['in_response_to_tweet_id'])
outbound_targeted['in_response_to_tweet_id'] = outbound_targeted['in_response_to_tweet_id'].astype('int64')

# 3. Build {parent_tweet_id -> company} and attach it to the inbound tweets it points back to.
reply_map = dict(zip(outbound_targeted['in_response_to_tweet_id'], outbound_targeted['author_id']))
inbound['company'] = inbound['tweet_id'].map(reply_map)

print(f"Built reply_map with {len(reply_map):,} entries.")
print(f"Inbound tweets total: {len(inbound):,}; with a recovered company: {inbound['company'].notna().sum():,}")

### Lab 1: Keep the tweets we recovered a company for

The demo above attached a `company` to every inbound tweet it could. Most inbound tweets have no recovered company (their reply was not in our subsample, or came from a different account). For the spot-check later we only want the rows that DID get a company.

**Your task** (about 5 minutes):

1. Filter `inbound` down to rows where `company` is not null. Call it `labeled`.
2. Keep only the columns `tweet_id`, `text`, `company` and reset the index.

Hints:

- A null check on a column gives you a boolean mask you can use to index the DataFrame.
- Selecting a list of columns and renumbering the rows are two short pandas calls; check the pandas docs if you are unsure which.

If you get stuck, expand the "Reveal the safety-net" panel below the verification cell.

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
# 1. notna() returns a boolean mask (True where company is set); indexing inbound with it
#    keeps just those rows. .copy() avoids a SettingWithCopyWarning on the next assignment.
labeled = inbound[inbound['company'].notna()].copy()

# 2. Double brackets select a list of columns; reset_index(drop=True) renumbers 0..n-1 and
#    drops the gappy index so later .iloc / .sample line up cleanly.
labeled = labeled[['tweet_id', 'text', 'company']].reset_index(drop=True)
```
</details>

In [ ]:
# 1. Keep only inbound tweets that received a recovered company label.
# notna() returns a boolean mask (True where company is set); indexing inbound with it
# keeps just those rows. .copy() avoids a SettingWithCopyWarning on the next assignment.
labeled = inbound[inbound['company'].notna()].copy()

# 2. Keep just the columns we need and renumber the rows.
# Double brackets select a list of columns; reset_index(drop=True) renumbers 0..n-1 and
# throws away the old (gappy) index so later .iloc / .sample line up cleanly.
labeled = labeled[['tweet_id', 'text', 'company']].reset_index(drop=True)
# Common mistake: forgetting reset_index leaves holes in the index from the filtered-out rows.

# Verification: count and plot the weak-label distribution.
if labeled is not None:
    print(f"Labeled tweets: {len(labeled):,}")
    counts = labeled['company'].value_counts()
    print(counts)
    plt.figure(figsize=(7, 3.5))
    counts.plot(kind='bar', color=['#5B8DEF', '#FF9F40', '#4ECDC4', '#FFD166'])
    plt.title('Recovered weak labels per company')
    plt.ylabel('Count'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# Silent rescue for Lab 1: if `labeled` is still None (lab skipped), back-fill a working
# value so the rest of the notebook runs. Does nothing once the lab is filled in. The full
# answer lives in the collapsed "Reveal the safety-net" panel above, not here.
if labeled is None:
    labeled = inbound[inbound['company'].notna()].copy()
    labeled = labeled[['tweet_id', 'text', 'company']].reset_index(drop=True)

## Section 3: Routing Categories and Light Cleaning

Here is the reframe that makes this a Part A capstone. The company that replied is *not* the routing decision. A real support inbox routes by **what the customer needs**: billing, technical help, account access, shipping, or general feedback. Those categories are our classification target, and we have zero labelled examples for them. That is exactly what zero-shot is for.

We define the categories as plain-English `candidate_labels`. Research on zero-shot is blunt about this: **label wording is the single biggest lever** on accuracy. Vague labels like "tech" or two labels that mean almost the same thing will hurt you; clear, distinct descriptions help.

Before routing we do **light** cleaning only. Transformer tokenizers tolerate moderate noise, so we do not strip everything. We do two things:

1. Remove URLs (they carry no routing signal and waste tokens).
2. Remove `@handles`. This matters: if `@AppleSupport` stays in the text, the model can cheat off the handle instead of reading the complaint, and in production customers often do not tag anyone.

In [ ]:
# Light cleaning: drop URLs and @handles, collapse whitespace. Keep case and punctuation;
# the NLI model reads natural English better than aggressively stripped text.
URL_RE = re.compile(r'http\S+|www\.\S+|t\.co/\S+')
HANDLE_RE = re.compile(r'@\w+')
WS_RE = re.compile(r'\s+')

def clean_tweet(text):
    if not isinstance(text, str):
        return ''
    text = URL_RE.sub(' ', text)
    text = HANDLE_RE.sub(' ', text)
    return WS_RE.sub(' ', text).strip()

# The routing categories: clear, distinct, human-readable phrases (wording matters a lot).
CANDIDATE_LABELS = [
    "billing or payment problem",
    "technical issue or bug",
    "account access or login",
    "shipping or delivery",
    "general feedback or praise",
]
# A hypothesis template tailored to the domain reads better to the NLI model than the default.
HYPOTHESIS_TEMPLATE = "This customer support message is about {}."

# Quick look at cleaning on 3 real tweets.
for ex in labeled['text'].iloc[:3] if labeled is not None else []:
    print('RAW  :', ex)
    print('CLEAN:', clean_tweet(ex))
    print()

### Lab 2: Clean every tweet and sample a routing batch

Zero-shot is slower per tweet than a trained model (each tweet runs once per candidate label), so in class we route a sample. The method is identical at full scale.

**Your task** (about 8 minutes):

1. Apply `clean_tweet` to `labeled['text']` and store the result in `labeled['clean']`.
2. Drop rows whose `clean` text is an empty string (some tweets were nothing but a link/handle).
3. Sample `SAMPLE_SIZE` rows (use `random_state=SEED`) into `routing_set` and reset the index.

Hints:

- pandas has a Series method that runs a function over every row.
- An empty string has length zero, which makes a clean boolean mask for the drop step.
- The DataFrame sampling method takes a count and a `random_state`; pass `SEED` so the draw is reproducible, then renumber the rows.

If you get stuck, expand the "Reveal the safety-net" panel below the verification cell.

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
# 1. Series.apply runs clean_tweet on each element and returns a new Series.
labeled['clean'] = labeled['text'].apply(clean_tweet)

# 2. str.len() > 0 is a boolean mask; some tweets were nothing but a link/handle and clean
#    to ''. reset_index(drop=True) renumbers the survivors before we sample.
labeled = labeled[labeled['clean'].str.len() > 0].reset_index(drop=True)

# 3. random_state=SEED makes the draw reproducible across runs and between students.
routing_set = labeled.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
```
</details>

In [ ]:
SAMPLE_SIZE = 200  # small enough to route in a minute or two in class

# 1. Clean every tweet.
# Series.apply runs clean_tweet on each element and returns a new Series we store as 'clean'.
labeled['clean'] = labeled['text'].apply(clean_tweet)

# 2. Drop rows whose cleaned text is empty.
# str.len() > 0 is a boolean mask; some tweets were nothing but a link/handle and clean to ''.
# reset_index(drop=True) again so the surviving rows are renumbered before we sample.
labeled = labeled[labeled['clean'].str.len() > 0].reset_index(drop=True)

# 3. Sample a routing batch.
# random_state=SEED makes the draw reproducible across runs. We never train, so this sample
# is purely "which 200 tweets do we route in class" - the method is identical at full scale.
routing_set = labeled.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
# Common mistake: sampling without random_state gives a different batch every run, so the
# verification numbers below would not match between students.

# Verification
if routing_set is not None:
    print(f"Routing set size: {len(routing_set):,}")
    print("Example cleaned tweet:", routing_set['clean'].iloc[0])

In [ ]:
# Silent rescue for Lab 2: `routing_set` feeds every routing and evaluation cell below. If the
# lab was skipped, back-fill `labeled` and a `routing_set` sample so the rest runs. Does nothing
# once the lab is filled in. The full answer is in the collapsed panel above.
if routing_set is None:
    if labeled is None:
        labeled = inbound[inbound['company'].notna()].copy()
        labeled = labeled[['tweet_id', 'text', 'company']].reset_index(drop=True)
    if 'clean' not in labeled.columns:
        labeled['clean'] = labeled['text'].apply(clean_tweet)
        labeled = labeled[labeled['clean'].str.len() > 0].reset_index(drop=True)
    routing_set = labeled.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)

## Section 4: Route Tweets With Zero-Shot Classification

This is the heart of the capstone. We load one pipeline and route any tweet to any of our categories, with no training and no labelled data. This `clf` is the same zero-shot classifier you met in A2 (there it was the `router`); here we point it at sharper support-routing labels:

```python
clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=DEVICE)
result = clf("my payment was charged twice this month",
             candidate_labels=CANDIDATE_LABELS,
             hypothesis_template=HYPOTHESIS_TEMPLATE)
```

The pipeline returns a dict whose `labels` and `scores` are **already sorted best-first**, so `result['labels'][0]` is the predicted route and `result['scores'][0]` is its confidence. No manual `argmax` needed. The model behind the scenes is an NLI model: it treats your tweet as a premise and each candidate label, plugged into the hypothesis template, as a hypothesis, then scores entailment.

**Figure: how zero-shot routing works as an NLI premise/hypothesis problem.**

```mermaid
graph TD
    A[Tweet text becomes premise] --> C[NLI model]
    B[Candidate label in hypothesis template] --> C
    C --> D[Entailment score per label]
    D --> E[Scores sorted best first]
    E --> F[labels 0 is predicted route]
    E --> G[scores 0 is confidence]
```


In [ ]:
# Load the zero-shot pipeline once (downloads facebook/bart-large-mnli on first run).
clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=DEVICE)

# Route a single, obviously-billing tweet to see the output shape.
demo_tweet = "I was charged twice for my subscription this month and want a refund"
result = clf(demo_tweet, candidate_labels=CANDIDATE_LABELS, hypothesis_template=HYPOTHESIS_TEMPLATE)

print("Tweet:", demo_tweet)
print("Predicted route:", result['labels'][0], f"(confidence {result['scores'][0]:.2f})")
print("\nFull ranking:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {score:.3f}  {label}")

### Lab 3: Route the entire routing batch

Now route every tweet in `routing_set`. Passing a list of texts (instead of one at a time) lets the pipeline batch them, which is much faster on GPU.

**Your task** (about 12 minutes):

1. Call `clf` once on the whole list of cleaned tweets, passing the same `candidate_labels` and `hypothesis_template` you used in the demo, plus a `batch_size` so it processes them in groups. Store the list of result dicts in `results`.
2. Each result dict ranks the labels best-first (same shape as the demo's single result). Pull the top label and its score for every tweet into two new columns `routing_set['route']` and `routing_set['confidence']`.

Hints:

- The demo above showed the single-tweet call shape; the batched call is the same but with a list of texts and a `batch_size` keyword. The output is a list of dicts in input order.
- A list comprehension over `results` keeps the order aligned with `routing_set`.

If you get stuck, expand the "Reveal the safety-net" panel below the verification cell.

<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
# 1. One batched call: batch_size=16 groups 16 tweets per forward pass, faster than one at a
#    time on GPU. The result is a list of dicts, one per input, in the SAME input order.
results = clf(routing_set['clean'].tolist(),
              candidate_labels=CANDIDATE_LABELS,
              hypothesis_template=HYPOTHESIS_TEMPLATE,
              batch_size=16)

# 2. labels/scores inside each dict are sorted best-first, so index [0] is the winning route
#    and its score. The comprehensions stay aligned with routing_set's row order.
routing_set['route'] = [r['labels'][0] for r in results]
routing_set['confidence'] = [r['scores'][0] for r in results]
```
</details>

In [ ]:
# 1. Route every tweet in the sample (this is the slow cell; a minute or two on GPU).
# Passing a list of texts lets the pipeline batch them; batch_size=16 groups 16 tweets per
# forward pass, which is faster than one-at-a-time on GPU. The result is a list of dicts,
# one per input text, in the SAME order as the input list.
results = clf(routing_set['clean'].tolist(),
              candidate_labels=CANDIDATE_LABELS,
              hypothesis_template=HYPOTHESIS_TEMPLATE,
              batch_size=16)

# 2. Pull the top label and its confidence into new columns.
# labels and scores inside each dict are already sorted best-first, so index [0] is the
# winning route and its score. A list comprehension keeps the order aligned with routing_set.
routing_set['route'] = [r['labels'][0] for r in results]
routing_set['confidence'] = [r['scores'][0] for r in results]
# Common mistake: calling clf once per row in a Python loop - far slower than one batched call.

# Verification: distribution of routes + a few routed examples.
if routing_set['route'].notna().any():
    print("Routes assigned:")
    print(routing_set['route'].value_counts())
    print("\nSpot-check:")
    for _, row in routing_set.head(5).iterrows():
        print(f"[{row['route']}  {row['confidence']:.2f}]  {row['clean'][:90]}")

In [ ]:
# Silent rescue for Lab 3: the confidence gate and packaging cells read routing_set['route']
# and routing_set['confidence']. If Lab 3 was skipped those columns are missing or all-None,
# so we route the sample here. Does real work only when the lab was not filled in; the full
# answer is in the collapsed panel above.
if 'route' not in routing_set.columns or routing_set['route'].isna().all():
    _results = clf(routing_set['clean'].tolist(),
                   candidate_labels=CANDIDATE_LABELS,
                   hypothesis_template=HYPOTHESIS_TEMPLATE,
                   batch_size=16)
    routing_set['route'] = [r['labels'][0] for r in _results]
    routing_set['confidence'] = [r['scores'][0] for r in _results]

## Section 5: Add Routing Metadata With NER

Routing decides the *queue*; metadata makes the ticket actionable. The agent who picks up a tweet wants to see at a glance which product or organization it mentions. We get that for free with the NER pipeline you already met in A2.

`aggregation_strategy="simple"` merges sub-word pieces into clean spans, so you get `iPhone`, not `i ##Phone`. The demo below loads the pipeline and shows the result shape; in the lab you will turn that raw output into a tidy list of org/product mentions.

### Lab 4 (Tier 3): Extract org and product mentions

This is the Part A deliverable's one no-scaffolding task. You get a function signature and a contract; there are no numbered steps, no `None` placeholders, and no in-cell hints. Read the demo output above, decide which entity groups matter, and implement the body yourself.


In [ ]:
# Demo: load the NER pipeline and look at its raw output shape (the call shape only).
# You will decide which of these entity_group values to keep when you write the lab below.
ner = pipeline("ner", aggregation_strategy="simple", device=DEVICE)

print(ner("my iPhone from Amazon stopped charging and I want a refund"))


In [ ]:
def extract_entities(text):
    """Run the NER pipeline on `text` and return the org/product-style mentions.

    Return a list of the surface strings (the 'word' field) for every entity that names an
    organization or a product. Inspect the demo output above to see which `entity_group`
    values your NER model actually emits, and remember different models tag products under
    different group names. A tweet with no such entities should return an empty list.
    """
    # ner(text) returns a list of dicts, each with 'entity_group' and 'word'.
    ents = ner(text)
    # Different models tag products as ORG, MISC, or PRODUCT, so we accept the whole family.
    # dict.get is safe if a key is ever missing; the comprehension yields [] when nothing matches.
    keep_groups = {"ORG", "MISC", "PRODUCT"}
    return [e["word"] for e in ents if e.get("entity_group") in keep_groups]


<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
def extract_entities(text):
    # ner(text) returns a list of dicts, each with 'entity_group' and 'word'.
    ents = ner(text)
    # Different models tag products as ORG, MISC, or PRODUCT, so accept the whole family.
    # dict.get is safe if a key is ever missing; the comprehension yields [] when nothing matches.
    keep_groups = {"ORG", "MISC", "PRODUCT"}
    return [e["word"] for e in ents if e.get("entity_group") in keep_groups]
```
</details>

In [ ]:
# Silent rescue for Lab 4: route_tweet (the packaging demo) calls extract_entities. If the lab
# was skipped, extract_entities may be undefined or return None, so we install a working
# fallback. Does nothing once the lab is filled in; the full answer is in the collapsed panel
# above (or, since this is a Tier 3 task, in the solution notebook).
try:
    extract_entities
except NameError:
    extract_entities = None

if extract_entities is None or extract_entities("my iPhone from Amazon broke") is None:
    if ner is None:
        ner = pipeline("ner", aggregation_strategy="simple", device=DEVICE)
    def extract_entities(text):
        ents = ner(text)
        return [e["word"] for e in ents if e.get("entity_group") in {"ORG", "MISC", "PRODUCT"}]

# Verification: route + entities for the first 20 tweets (works whether or not the lab was filled).
for _, row in routing_set.head(20).iterrows():
    ents = extract_entities(row['clean'])
    print(f"[{row['route']}] entities={ents}  ::  {row['clean'][:80]}")


## Section 6: Evaluate and Add a Safety Net

Two questions remain: how good is the routing, and what do we do when the model is unsure?

**Evaluation.** We have no true routing labels, but we have the weak `company` labels. They do not map one-to-one to our categories, so a raw accuracy number would be misleading. Instead we do two honest things: a qualitative spot-check (read routed examples), and a confidence histogram to see how decisive the model is.

**Confidence fallback.** Production routers do not auto-route everything. The standard pattern is: accept high-confidence predictions, and **abstain** below a threshold, sending those tweets to a human queue. The confidence score is the knob that trades human effort for accuracy.

**Choosing the threshold.** Look at the histogram you are about to plot. With five candidate labels, a score near 0.2 is barely above chance, so a tweet the model scores below about 0.5 is one it is genuinely unsure about. We start at `0.5` as a reasonable first cut and let you move it: a higher threshold escalates more tweets (safer, more human work), a lower one auto-routes more (cheaper, riskier). The homework asks you to sweep it.

### Lab 5: Confidence threshold and route-or-escalate

**Your task** (about 10 minutes):

1. Plot a histogram of `routing_set['confidence']`.
2. Set `CONF_THRESHOLD = 0.5`. Create a `final_route` column equal to the predicted `route` when `confidence >= CONF_THRESHOLD`, otherwise the string `"ESCALATE_TO_HUMAN"`.
3. Print how many tweets were auto-routed vs escalated.

**Figure: the confidence gate that auto-routes or escalates to a human.**

```mermaid
flowchart LR
    A[Top label confidence] --> B{confidence >= CONF_THRESHOLD}
    B -->|yes| C[Auto-route to predicted queue]
    B -->|no| D[ESCALATE_TO_HUMAN]
    D --> E[Human review queue]
```


<details>
<summary>Stuck? Reveal the safety-net</summary>

```python
# np.where(cond, a, b) is a vectorized if/else over the whole column: where confidence clears
# the bar keep the predicted route, otherwise replace it with the escalation sentinel.
CONF_THRESHOLD = 0.5
routing_set['final_route'] = np.where(
    routing_set['confidence'] >= CONF_THRESHOLD,
    routing_set['route'],
    "ESCALATE_TO_HUMAN")
```
</details>

In [ ]:
# 1. Confidence histogram (read it to sanity-check the 0.5 threshold below).
plt.figure(figsize=(6, 3))
plt.hist(routing_set['confidence'], bins=20, color='#5B8DEF')
plt.title('Zero-shot routing confidence'); plt.xlabel('top-label score'); plt.ylabel('count')
plt.tight_layout(); plt.show()

# 2. Apply a confidence threshold: keep the route when confident, else escalate.
# np.where(cond, a, b) is a vectorized if/else over the whole column: where confidence clears
# the bar we keep the predicted route, otherwise we replace it with the escalation sentinel.
# This is the production pattern: trade a little coverage for higher precision on what we auto-route.
CONF_THRESHOLD = 0.5
routing_set['final_route'] = np.where(
    routing_set['confidence'] >= CONF_THRESHOLD,
    routing_set['route'],
    "ESCALATE_TO_HUMAN")

# Silent rescue: if the lab is left empty, fill final_route so the report below still runs.
# The full answer is in the collapsed "Reveal the safety-net" panel above.
if routing_set['final_route'].isna().all():
    routing_set['final_route'] = np.where(
        routing_set['confidence'] >= CONF_THRESHOLD,
        routing_set['route'],
        "ESCALATE_TO_HUMAN")

# 3. Report the auto vs escalate split.
escalated = (routing_set['final_route'] == "ESCALATE_TO_HUMAN").sum()
print(f"Auto-routed: {len(routing_set) - escalated:,}")
print(f"Escalated to human: {escalated:,}")
print("\nFinal route distribution:")
print(routing_set['final_route'].value_counts())

**Figure: the end-to-end route_tweet pipeline from raw tweet to routed ticket.**

```mermaid
graph TD
    A[Raw customer tweet] --> B[clean_tweet drop URLs and handles]
    B --> C[zero-shot classify with candidate_labels]
    C --> D[Top route and confidence]
    B --> E[NER extract org and product entities]
    D --> F[Confidence gate]
    E --> G[Routing metadata]
    F --> H[Routed ticket with metadata]
    G --> H
```


In [ ]:
# Package everything into one function: clean -> route -> entities -> confidence gate.
# extract_entities was written and verified in Lab 4 above; we reuse it here unchanged.
def route_tweet(text, threshold=CONF_THRESHOLD):
    clean = clean_tweet(text)
    r = clf(clean, candidate_labels=CANDIDATE_LABELS, hypothesis_template=HYPOTHESIS_TEMPLATE)
    route = r['labels'][0]
    conf = r['scores'][0]
    final = route if conf >= threshold else "ESCALATE_TO_HUMAN"
    entities = extract_entities(clean)  # reuse the Lab 4 helper, do not re-implement it
    return {"final_route": final, "confidence": round(conf, 3), "entities": entities}

# Try it on a fresh, untagged complaint (no @handle, like real production traffic).
print(route_tweet("my package never arrived and the tracking link is dead"))

# Optional live demo: a tiny Gradio app. Guarded so the notebook still runs without gradio.
# This is the SAME UX skeleton the Part C chatbot will use - just a different model behind it.
try:
    import gradio as gr
    def _ui(text):
        out = route_tweet(text)
        return f"Route: {out['final_route']} (conf {out['confidence']})\nEntities: {out['entities']}"
    demo = gr.Interface(fn=_ui, inputs=gr.Textbox(label="Customer tweet"),
                        outputs=gr.Textbox(label="Routing decision"), title="Support Router (zero-shot)")
    # demo.launch(share=True)  # uncomment in Colab to get a public link
    print("Gradio app defined. Uncomment demo.launch(share=True) to open it.")
except ImportError:
    print("gradio not installed - skipping the live demo (the route_tweet function still works).")

## Wrap-up: You Shipped a Router With Zero Training

Look back at what you just did. With no labelled training set and no training loop, you:

- recovered weak labels from implicit reply metadata,
- defined routing categories as plain-English `candidate_labels`,
- routed real, noisy tweets with `pipeline("zero-shot-classification")`,
- enriched each ticket with NER metadata,
- added a confidence threshold that escalates unsure tweets to a human,
- and wrapped it all in a tiny Gradio app.

**Reflect:**

1. Would you ship this today? For a brand-new problem with no data, very possibly yes - zero-shot is a real, deployable baseline.
2. Where does it stop? We did not measure a clean accuracy here on purpose: the weak `company` labels are not routing ground truth, so any single number would be misleading (that is why we spot-checked and looked at the confidence histogram instead). But the well-known pattern holds: a zero-shot baseline gets you a usable router fast, and a transformer fine-tuned on real labelled routing data, evaluated properly, reliably beats it. For high-volume routing even a few points of accuracy is a lot of tickets.
3. What would push past the ceiling? A stronger zero-shot backbone helps a little (`MoritzLaurer/deberta-v3-base-mnli-fever-anli` beats `bart-large-mnli`). But the real jump is **fine-tuning your own classifier** on labelled data - and measuring it with real metrics (precision, recall, F1) against a held-out set.

**The bridge to Part B.** To fine-tune a transformer you first have to understand what is inside one: tensors, embeddings, neural-network layers, and the training loop. That is exactly Part B, starting with PyTorch tensors in B4. By Part C you will fine-tune DistilBERT and serve it in this same Gradio shell - your zero-shot router, upgraded into a trained one.

## Homework Extension (async, deeper)

Pick one or more. These are genuinely harder than the in-class labs, not just more volume.

1. **Multi-label routing.** Some tweets need two queues ("I was double-charged AND the app crashed"). Re-run the router with `multi_label=True`, then surface every label whose score is above a secondary threshold (e.g. 0.4) as additional routes. Decide how an agent should see a primary plus secondary route.
2. **Threshold tuning against weak labels.** Build a crude mapping from your `company` weak labels to plausible categories, then sweep `CONF_THRESHOLD` from 0.3 to 0.8 and plot escalation rate vs apparent agreement. Where is the sensible operating point?
3. **Model upgrade.** Swap `facebook/bart-large-mnli` for `MoritzLaurer/deberta-v3-base-mnli-fever-anli` and compare routes and confidences on the same sample. Did the harder cases improve? At what speed cost?
4. **Ship it.** Push the `route_tweet` Gradio app to a free HuggingFace Space (app.py + requirements.txt with `transformers` and `torch`). This is the deployment muscle you will reuse for the Part C chatbot.